# syft-ingest API Exploration

## 1. Environment check

In [2]:
import os

token = os.getenv("BRIGHTDATA_API_TOKEN")
print(
    f"BRIGHTDATA_API_TOKEN : {token[:3] + '...' + token[-3:] if token else 'NOT SET -- BrightData cells will fail'}"
)

BRIGHTDATA_API_TOKEN : 7a5...1d5


## 2. Core imports

In [ ]:
import sys
from pathlib import Path

import syft_ingest
from syft_ingest import FetchConfig, gather

# Add parent directory to Python path so imports work
notebook_dir = Path(".").resolve()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

print(f"✅ Added to sys.path: {notebook_dir}")
print("✅ Imports OK")

## 3. Simplified gather() API — YouTube video

In [ ]:
# Simplest case: just platform + URLs
corpus = gather(
    "youtube", ["https://www.youtube.com/watch?v=kCc8FmEb1nY"]
)  # Karpathy: GPT from scratch

print(f"Items fetched: {len(corpus.all_items())}")
if corpus.all_items():
    item = corpus.all_items()[0]
    print(f"  Title: {item.title}")
    print(f"  Author: {item.author}")

## 4. Channel enumeration with config

In [ ]:
# With config options passed as kwargs
corpus = gather(
    "youtube",
    ["https://www.youtube.com/@AndrejKarpathy"],
    playlistend=5,  # Cap at 5 videos to keep it fast
)

print(f"Videos found: {len(corpus.all_items())}")
for item in corpus.all_items():
    print(f"  [{item.published_at}] {item.title}")

## 5. Facebook scraping

Requires `BRIGHTDATA_API_TOKEN`. Triggers a real scrape job and polls until done (~60-120s).

In [ ]:
corpus = gather(
    "facebook",
    ["https://www.facebook.com/karpathy"],
    author="Andrej Karpathy",
    timeout=120,
    poll_interval=5,
)

print(f"Items fetched: {len(corpus.all_items())}")
print(f"Author: {corpus.person}")

if corpus.all_items():
    item = corpus.all_items()[0]
    print("\nFirst item:")
    print(f"  Type: {type(item).__name__}")
    print(f"  Title: {item.title}")
    print(f"  Text: {(item.text or '')[:120]}")

## 6. Instagram scraping with posts_limit for testing

In [ ]:
corpus = gather(
    "instagram",
    ["https://www.instagram.com/karpathy/"],
    author="Andrej Karpathy",
    timeout=120,
    poll_interval=5,
    posts_limit=5,  # Limit to 5 posts for quick testing (no posts_limit = all posts)
)

print(f"Items fetched: {len(corpus.all_items())}")
for i, item in enumerate(corpus.all_items()[:3], 1):
    print(f"\n{i}. {type(item).__name__}: {item.title}")

## 7. Error handling and validation

In [ ]:
# Invalid platform raises ValueError
try:
    corpus = gather("tiktok", ["https://tiktok.com/user"])
except ValueError as e:
    print(f"✅ ValueError caught for unsupported platform: {e}")

# Missing URLs raises ValueError
try:
    corpus = gather("youtube", None)
except ValueError as e:
    print(f"✅ ValueError caught for missing URLs: {e}")

## 8. Configuration validation with FetchConfig

In [ ]:
# FetchConfig provides validation and IDE autocomplete
# All these options are validated:

config = FetchConfig(
    # YouTube options
    socket_timeout=60,  # Network timeout (seconds)
    playlistend=10,  # Max videos from channel
    download_full_video=False,  # Enable full video download
    # BrightData options
    timeout=300,  # Scrape job timeout
    poll_interval=2,  # Check frequency
    posts_limit=5,  # Limit posts (for testing)
)

print("✅ FetchConfig created with validation:")
print(f"  socket_timeout: {config.socket_timeout}")
print(f"  posts_limit: {config.posts_limit}")
print()

# Invalid values are caught immediately
try:
    bad = FetchConfig(socket_timeout=-1)  # Must be >= 1
except Exception as e:
    print(f"✅ Validation error (expected): {type(e).__name__}")

In [ ]:
# Simplest: just platform + URLs
corpus = syft_ingest.gather("youtube", ["https://youtube.com/watch?v=..."])

# With author
corpus = syft_ingest.gather(
    "youtube", ["https://youtube.com/watch?v=..."], author="Andrej"
)

# With config options
corpus = syft_ingest.gather(
    "youtube", ["https://youtube.com/@user"], socket_timeout=60, playlistend=10
)

# With Instagram/Facebook (requires BRIGHTDATA_API_TOKEN)
corpus = syft_ingest.gather(
    "instagram", ["https://instagram.com/user/"], timeout=120, posts_limit=5
)

# Export results
# corpus.export("jsonl", output="out.jsonl")

print("✅ API ready for use!")